# Reproduction notebook for arXiv:2603.06431 (Lp + W1p only)

This notebook reproduces the paper-style experiments for **Lp** and **W1p** and intentionally excludes **W2p**.

⚠️ Stability note: this version uses **chunked Monte Carlo** for W1p and configurable quick settings to avoid kernel OOM/kill.


In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / "src" / "intervalnets").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

import gc
import math
import random
from dataclasses import dataclass

import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

from intervalnets import IntervalTensor, enable_interval_eval

enable_interval_eval()
torch.set_num_threads(max(1, min(torch.get_num_threads(), 8)))


## Configuration

In [ ]:
BASE_SEED = 1234

# Keep QUICK_MODE=True by default for stability in notebooks.
QUICK_MODE = True

if QUICK_MODE:
    N_RUNS = 8
    ITERATIONS = list(range(0, 7))
    EPOCHS_1D = 150
    EPOCHS_2D_LP = 200
    EPOCHS_2D_W1P = 300
    MC_REF_SAMPLES_1D = 8_000
    MC_REF_SAMPLES_2D = 10_000
    MC_BATCH = 512
    DEEP_WIDTH = 10
    WIDE_WIDTH = 100
else:
    # Paper-like heavier settings
    N_RUNS = 100
    ITERATIONS = list(range(0, 11))
    EPOCHS_1D = 2000
    EPOCHS_2D_LP = 2000
    EPOCHS_2D_W1P = 10_000
    MC_REF_SAMPLES_1D = 50_000
    MC_REF_SAMPLES_2D = 50_000
    MC_BATCH = 2048
    DEEP_WIDTH = 32
    WIDE_WIDTH = 200

P_VAL = 2.0
print(f"QUICK_MODE={QUICK_MODE}, runs={N_RUNS}, iterations={len(ITERATIONS)}")
print(f"widths: deep=3x{DEEP_WIDTH}, wide=1x{WIDE_WIDTH}")

## Helpers

In [ ]:
@dataclass
class Arch:
    name: str
    input_dim: int
    hidden_layers: int
    width: int
    activation: str


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def make_network(arch: Arch) -> nn.Sequential:
    act = nn.Tanh if arch.activation == "tanh" else nn.ReLU
    layers = []
    in_dim = arch.input_dim
    for _ in range(arch.hidden_layers):
        layers.append(nn.Linear(in_dim, arch.width))
        layers.append(act())
        in_dim = arch.width
    layers.append(nn.Linear(in_dim, 1))
    return nn.Sequential(*layers)


def ci95(x: np.ndarray):
    m = x.mean(axis=0)
    if x.shape[0] <= 1:
        return m, m, m
    s = x.std(axis=0, ddof=1)
    h = 1.96 * s / np.sqrt(x.shape[0])
    return m, m - h, m + h


def gaussian_peak_1d(x: torch.Tensor) -> torch.Tensor:
    return torch.exp(-40.0 * x.pow(2))


def smooth_disk_2d(xy: torch.Tensor, radius: float = 0.6) -> torch.Tensor:
    r2 = xy[:, 0].pow(2) + xy[:, 1].pow(2)
    out = torch.zeros_like(r2)
    inside = r2 < radius * radius
    t = 1.0 - r2[inside] / (radius * radius)
    out[inside] = torch.exp(-1.0 / torch.clamp(t, min=1e-8))
    return out


def train_to_target(model: nn.Module, dim: int, target_fn, epochs: int, lr: float = 1e-3, batch_size: int = 1024):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for _ in range(epochs):
        x = torch.rand(batch_size, dim) * 2.0 - 1.0
        y = target_fn(x if dim > 1 else x[:, :1])
        pred = model(x).squeeze(-1)
        loss = ((pred - y) ** 2).mean()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()


def mc_lp(model: nn.Module, dim: int, p: float, n: int, batch: int) -> float:
    model.eval()
    total = 0.0
    seen = 0
    with torch.no_grad():
        while seen < n:
            m = min(batch, n - seen)
            x = torch.rand(m, dim) * 2.0 - 1.0
            y = model(x).squeeze(-1).abs().pow(p)
            total += float(y.sum().item())
            seen += m
    integral = (2.0 ** dim) * total / n
    return integral ** (1.0 / p)


def mc_w1p(model: nn.Module, dim: int, p: float, n: int, batch: int) -> float:
    # Chunked to prevent kernel OOM from huge requires_grad tensors.
    model.eval()
    total = 0.0
    seen = 0
    while seen < n:
        m = min(batch, n - seen)
        x = torch.rand(m, dim, requires_grad=True) * 2.0 - 1.0
        y = model(x).squeeze(-1)
        grad = torch.autograd.grad(y.sum(), x, create_graph=False, retain_graph=False)[0]
        integrand = y.abs().pow(p) + torch.linalg.vector_norm(grad, ord=2, dim=-1).pow(p)
        total += float(integrand.detach().sum().item())
        seen += m
        del x, y, grad, integrand
    integral = (2.0 ** dim) * total / n
    return integral ** (1.0 / p)


def bound_gap_curve(model: nn.Module, domain: IntervalTensor, p: float, mode: str, iterations: list[int], ref_value: float) -> np.ndarray:
    out = []
    for it in iterations:
        if mode == "lp":
            b = model.lpnorm(domain, p=p, iterations=it)
        else:
            b = model.sobolev_norm(domain, p=p, iterations=it)
        out.append((float(b.upper - b.lower)) / max(ref_value, 1e-12))
    return np.array(out, dtype=float)


## 1D setups

In [ ]:
deep_tanh_1d = Arch("deep", 1, 3, DEEP_WIDTH, "tanh")
wide_tanh_1d = Arch("wide", 1, 1, WIDE_WIDTH, "tanh")

deep_relu_1d = Arch("deep", 1, 3, DEEP_WIDTH, "relu")
wide_relu_1d = Arch("wide", 1, 1, WIDE_WIDTH, "relu")

domain_1d = IntervalTensor.from_bounds([-1.0], [1.0])


def run_family(arch: Arch, mode: str, trained: bool):
    curves = []
    for run in range(N_RUNS):
        set_seed(BASE_SEED + run)
        model = make_network(arch)
        if trained:
            train_to_target(model, dim=1, target_fn=gaussian_peak_1d, epochs=EPOCHS_1D)

        if mode == "w1p":
            ref = mc_w1p(model, dim=1, p=P_VAL, n=MC_REF_SAMPLES_1D, batch=MC_BATCH)
        else:
            ref = mc_lp(model, dim=1, p=P_VAL, n=MC_REF_SAMPLES_1D, batch=MC_BATCH)

        curves.append(bound_gap_curve(model, domain_1d, p=P_VAL, mode=mode, iterations=ITERATIONS, ref_value=ref))
        del model
        gc.collect()

    return np.stack(curves, axis=0)


## Figure A — 1D W1p (untrained vs trained)

In [ ]:
w1p_deep_untrained = run_family(deep_tanh_1d, mode="w1p", trained=False)
print("untrained 1/2 done")
w1p_wide_untrained = run_family(wide_tanh_1d, mode="w1p", trained=False)
print("untrained done")

w1p_deep_trained = run_family(deep_tanh_1d, mode="w1p", trained=True)
print("trained 1/2 done")
w1p_wide_trained = run_family(wide_tanh_1d, mode="w1p", trained=True)
print("trained done")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, deep_arr, wide_arr, title in [
    (axes[0], w1p_deep_untrained, w1p_wide_untrained, "Untrained tanh networks"),
    (axes[1], w1p_deep_trained, w1p_wide_trained, "Trained tanh networks (Gaussian peak)"),
]:
    for label, arr, color in [(f"deep (3x{DEEP_WIDTH})", deep_arr, "tab:blue"), (f"wide (1x{WIDE_WIDTH})", wide_arr, "tab:orange")]:
        m, lo, hi = ci95(arr)
        ax.plot(ITERATIONS, m, color=color, label=label)
        ax.fill_between(ITERATIONS, lo, hi, color=color, alpha=0.2)
    ax.set_yscale("log")
    ax.set_xlabel("refinement iterations")
    ax.grid(True, alpha=0.3)
    ax.set_title(title)

axes[0].set_ylabel("normalized global bound gap")
axes[0].legend()
fig.suptitle("1D W1p reproduction")
plt.tight_layout()
plt.show()

## Figure B — 1D Lp (untrained vs trained)

In [ ]:
lp_deep_untrained = run_family(deep_relu_1d, mode="lp", trained=False)
lp_wide_untrained = run_family(wide_relu_1d, mode="lp", trained=False)

lp_deep_trained = run_family(deep_relu_1d, mode="lp", trained=True)
lp_wide_trained = run_family(wide_relu_1d, mode="lp", trained=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, deep_arr, wide_arr, title in [
    (axes[0], lp_deep_untrained, lp_wide_untrained, "Untrained ReLU networks"),
    (axes[1], lp_deep_trained, lp_wide_trained, "Trained ReLU networks (Gaussian peak)"),
]:
    for label, arr, color in [(f"deep (3x{DEEP_WIDTH})", deep_arr, "tab:green"), (f"wide (1x{WIDE_WIDTH})", wide_arr, "tab:red")]:
        m, lo, hi = ci95(arr)
        ax.plot(ITERATIONS, m, color=color, label=label)
        ax.fill_between(ITERATIONS, lo, hi, color=color, alpha=0.2)
    ax.set_yscale("log")
    ax.set_xlabel("refinement iterations")
    ax.grid(True, alpha=0.3)
    ax.set_title(title)

axes[0].set_ylabel("normalized global bound gap")
axes[0].legend()
fig.suptitle("1D Lp reproduction")
plt.tight_layout()
plt.show()

## 2D trained experiments (Figure C + D)

In [ ]:
deep_relu_2d = Arch("deep", 2, 3, DEEP_WIDTH, "relu")
wide_relu_2d = Arch("wide", 2, 1, WIDE_WIDTH, "relu")
deep_tanh_2d = Arch("deep", 2, 3, DEEP_WIDTH, "tanh")
wide_tanh_2d = Arch("wide", 2, 1, WIDE_WIDTH, "tanh")
domain_2d = IntervalTensor.from_bounds([-1.0, -1.0], [1.0, 1.0])

set_seed(BASE_SEED + 1000)
lp_deep_2d = make_network(deep_relu_2d)
train_to_target(lp_deep_2d, dim=2, target_fn=smooth_disk_2d, epochs=EPOCHS_2D_LP)
lp_ref_deep = mc_lp(lp_deep_2d, dim=2, p=P_VAL, n=MC_REF_SAMPLES_2D, batch=MC_BATCH)
lp_curve_deep = bound_gap_curve(lp_deep_2d, domain_2d, p=P_VAL, mode="lp", iterations=ITERATIONS, ref_value=lp_ref_deep)

set_seed(BASE_SEED + 1001)
lp_wide_2d = make_network(wide_relu_2d)
train_to_target(lp_wide_2d, dim=2, target_fn=smooth_disk_2d, epochs=EPOCHS_2D_LP)
lp_ref_wide = mc_lp(lp_wide_2d, dim=2, p=P_VAL, n=MC_REF_SAMPLES_2D, batch=MC_BATCH)
lp_curve_wide = bound_gap_curve(lp_wide_2d, domain_2d, p=P_VAL, mode="lp", iterations=ITERATIONS, ref_value=lp_ref_wide)

set_seed(BASE_SEED + 1100)
w1p_deep_2d = make_network(deep_tanh_2d)
train_to_target(w1p_deep_2d, dim=2, target_fn=smooth_disk_2d, epochs=EPOCHS_2D_W1P)
w1_ref_deep = mc_w1p(w1p_deep_2d, dim=2, p=P_VAL, n=MC_REF_SAMPLES_2D, batch=MC_BATCH)
w1_curve_deep = bound_gap_curve(w1p_deep_2d, domain_2d, p=P_VAL, mode="w1p", iterations=ITERATIONS, ref_value=w1_ref_deep)

set_seed(BASE_SEED + 1101)
w1p_wide_2d = make_network(wide_tanh_2d)
train_to_target(w1p_wide_2d, dim=2, target_fn=smooth_disk_2d, epochs=EPOCHS_2D_W1P)
w1_ref_wide = mc_w1p(w1p_wide_2d, dim=2, p=P_VAL, n=MC_REF_SAMPLES_2D, batch=MC_BATCH)
w1_curve_wide = bound_gap_curve(w1p_wide_2d, domain_2d, p=P_VAL, mode="w1p", iterations=ITERATIONS, ref_value=w1_ref_wide)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].plot(ITERATIONS, lp_curve_deep, marker='o', label=f'deep (3x{DEEP_WIDTH})')
axes[0].plot(ITERATIONS, lp_curve_wide, marker='o', label=f'wide (1x{WIDE_WIDTH})')
axes[0].set_title('2D trained Lp (ReLU)')
axes[0].set_yscale('log')
axes[0].set_xlabel('refinement iterations')
axes[0].set_ylabel('normalized global bound gap')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(ITERATIONS, w1_curve_deep, marker='o', label=f'deep (3x{DEEP_WIDTH})')
axes[1].plot(ITERATIONS, w1_curve_wide, marker='o', label=f'wide (1x{WIDE_WIDTH})')
axes[1].set_title('2D trained W1p (tanh)')
axes[1].set_yscale('log')
axes[1].set_xlabel('refinement iterations')
axes[1].grid(True, alpha=0.3)
axes[1].legend()
plt.tight_layout(); plt.show()


def local_gap_heatmap(model: nn.Module, mode: str, grid_n: int = 20):
    xs = np.linspace(-1, 1, grid_n + 1)
    ys = np.linspace(-1, 1, grid_n + 1)
    out = np.zeros((grid_n, grid_n))
    for i in range(grid_n):
        for j in range(grid_n):
            box = IntervalTensor.from_bounds([float(xs[i]), float(ys[j])], [float(xs[i+1]), float(ys[j+1])])
            b = model.lpnorm(box, p=P_VAL, iterations=0) if mode == 'lp' else model.sobolev_norm(box, p=P_VAL, iterations=0)
            out[j, i] = float(b.upper - b.lower)
    return out

h_lp = local_gap_heatmap(lp_deep_2d, 'lp')
h_w1 = local_gap_heatmap(w1p_deep_2d, 'w1p')
fig, axs = plt.subplots(1,2,figsize=(10,4))
axs[0].imshow(h_lp, origin='lower', extent=[-1,1,-1,1], cmap='magma'); axs[0].set_title('Lp local gap (deep)')
axs[1].imshow(h_w1, origin='lower', extent=[-1,1,-1,1], cmap='magma'); axs[1].set_title('W1p local gap (deep)')
plt.tight_layout(); plt.show()

## Why kernels died before

The previous notebook used a single huge `requires_grad=True` tensor for W1p Monte Carlo reference estimates,
which can consume large memory and crash kernels. This notebook fixes that by batching/chunking W1p MC.
